# 02 — PyHealth Overview

PyHealth follows a consistent 4-step pipeline for all clinical prediction tasks:

```
Dataset  ──►  Task  ──►  Model  ──►  Trainer
   │             │           │           │
MIMIC/OMOP   set_task()   RETAIN     train()
SampleData   readmission  Transformer evaluate()
             mortality    SafeDrug
             drug_rec     MoleRec
```

This notebook runs a minimal end-to-end demo using synthetic data.

In [ ]:
from pyhealth_enterprise.config import settings

# Step 1: Load dataset
from pyhealth_enterprise.datasets.synthetic import SyntheticEHRDataset

ds = SyntheticEHRDataset()
ds.load()
ds.stat()

In [ ]:
# Step 2: Attach a clinical task
from pyhealth.tasks import readmission_prediction_mimic3_fn

task_dataset = ds.dataset.set_task(readmission_prediction_mimic3_fn)
task_dataset.stat()

In [ ]:
# Step 3: Split and create DataLoaders
from pyhealth.datasets import get_dataloader, split_by_patient

train, val, test = split_by_patient(task_dataset, [0.8, 0.1, 0.1])
train_loader = get_dataloader(train, batch_size=32, shuffle=True)
val_loader   = get_dataloader(val,   batch_size=32, shuffle=False)
test_loader  = get_dataloader(test,  batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")
print(f"Test batches : {len(test_loader)}")

In [ ]:
# Step 4a: Instantiate model
from pyhealth.models import RETAIN

model = RETAIN(
    dataset=task_dataset,
    feature_keys=["conditions", "drugs"],
    label_key="readmission",
    mode="binary",
)
print(model)

In [ ]:
# Step 4b: Train and evaluate
from pyhealth.trainer import Trainer

trainer = Trainer(model=model, metrics=["pr_auc", "roc_auc", "f1"])
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=5,        # small for demo; use 50+ for real experiments
    monitor="pr_auc",
)
result = trainer.evaluate(test_loader)
print("\nTest results:")
for metric, value in result.items():
    print(f"  {metric}: {value:.4f}")

## PyHealth Module Map

| Module | What it provides |
|---|---|
| `pyhealth.datasets` | MIMIC3, MIMIC4, OMOP, eICU, SampleDataset, split_by_patient, get_dataloader |
| `pyhealth.tasks` | readmission_, mortality_, los_, drug_recommendation_ task functions |
| `pyhealth.models` | RETAIN, Transformer, SafeDrug, MoleRec, GRASP, GAMENet |
| `pyhealth.trainer` | Trainer (train, evaluate, checkpoint) |
| `pyhealth.metrics` | binary_metrics_fn, multilabel_metrics_fn, ddi_rate_score |
| `pyhealth.medcode` | CrossMap (ICD9→ICD10, NDC→ATC), ATC lookup |

## Next: Explore Datasets →
Open `../02_datasets/01_synthetic_dataset_exploration.ipynb`